# speed_benchmark

**Question.** Which device and parallelism finishes runs fastest on this M1 Pro?

**Setup.** `configs/experiment/speed_benchmark.yaml` (shred, flow_trunk_shred_w256, flow_trunk_shred_div; 100 fixed epochs, seed 0), 6 identical jobs = 18 runs per setup, 2 repeats, driven by `experiments/speed_benchmark/bench.sh`. Record: `experiments/speed_benchmark/README.md`.

In [1]:
import csv, glob, json, statistics as st
from collections import defaultdict
from pathlib import Path

out = Path("../outputs/speed_benchmark")
spans = defaultdict(list)
for row in csv.DictReader(open(out / "timings.csv")):
    spans[row["setup"]].append(float(row["makespan_s"]))
base = 18 / st.median(spans["mps_1"])
for setup, s in sorted(spans.items(), key=lambda kv: st.median(kv[1])):
    rate = 18 / st.median(s) * 60
    print(f"{setup:8s} makespans {s}  {rate:5.1f} runs/min  {rate / base / 60:.1f}x vs mps_1")

hybrid   makespans [49.5, 51.1]   21.5 runs/min  4.7x vs mps_1
cpu_6x1  makespans [52.5, 53.2]   20.4 runs/min  4.5x vs mps_1
cpu_3x2  makespans [77.6, 75.6]   14.1 runs/min  3.1x vs mps_1
mps_3    makespans [96.2, 85.0]   11.9 runs/min  2.6x vs mps_1
cpu_2x3  makespans [98.4, 88.5]   11.6 runs/min  2.5x vs mps_1
mps_2    makespans [137.8, 128.5]    8.1 runs/min  1.8x vs mps_1
cpu_1x6  makespans [149.3, 140.9]    7.4 runs/min  1.6x vs mps_1
cpu_1x1  makespans [201.9, 192.7]    5.5 runs/min  1.2x vs mps_1
mps_1    makespans [249.2, 222.6]    4.6 runs/min  1.0x vs mps_1


In [2]:
wall, err = defaultdict(list), defaultdict(list)
for f in glob.glob(str(out / "*/r*/job*/nektar_uv/seed0/*/result.json")):
    d = json.load(open(f))
    method = d["run"].split("__")[0]
    wall[(f.split("/")[-7], d["device"], method)].append(d["wall_time_s"])
    err[(d["device"], method)].append(d["metrics"]["clean"]["err"])
for k in [k for k in sorted(wall) if k[0] in ("mps_1", "cpu_1x6")]:
    print(k, f"median wall {st.median(wall[k]):.1f} s")
for k in sorted(err):
    print(k, f"test err {st.median(err[k]):.4f} (range {max(err[k]) - min(err[k]):.1e})")

('cpu_1x6', 'cpu', 'flow_trunk_shred_div') median wall 11.7 s
('cpu_1x6', 'cpu', 'flow_trunk_shred_w256') median wall 4.8 s
('cpu_1x6', 'cpu', 'shred') median wall 4.1 s
('mps_1', 'mps', 'flow_trunk_shred_div') median wall 19.2 s
('mps_1', 'mps', 'flow_trunk_shred_w256') median wall 7.9 s
('mps_1', 'mps', 'shred') median wall 8.0 s
('cpu', 'flow_trunk_shred_div') test err 0.2303 (range 0.0e+00)
('cpu', 'flow_trunk_shred_w256') test err 0.2283 (range 4.5e-08)
('cpu', 'shred') test err 0.0104 (range 3.4e-07)
('mps', 'flow_trunk_shred_div') test err 0.2302 (range 0.0e+00)
('mps', 'flow_trunk_shred_w256') test err 0.2291 (range 0.0e+00)
('mps', 'shred') test err 0.0086 (range 0.0e+00)


**Result.** 6 CPU jobs × 1 thread: 20.4 runs/min (4.5× one MPS job); hybrid 2 MPS + 4 CPU: 21.5 runs/min (4.7×). CPU is faster than MPS even for a single job. Errors are deterministic per device but differ between devices (shred 0.0104 CPU vs 0.0086 MPS at 100 epochs), so do not mix devices within a comparison.